# Movie Popularity Analysis — IMDb + TMDb

**DSA 210 Term Project — Korcan Baykal**

This notebook covers data collection, cleaning, exploratory data analysis (EDA), and hypothesis testing on a merged IMDb + TMDb movie dataset.

**Table of Contents**
1. Data Loading
2. Filtering and Merging IMDb Tables
3. Cleaning and Type Conversion
4. Exploratory Data Analysis — IMDb
5. Data Enrichment with TMDb API
6. Exploratory Data Analysis — Merged IMDb + TMDb
7. Additional EDA — Log Scale and Genre Comparison
8. Hypothesis Tests
9. Summary of Findings

## 1. Data Loading

We load two IMDb TSV files (`title.basics` and `title.ratings`) that contain movie metadata and user ratings respectively.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# ── Google Colab compatibility ────────────────────────────────────────────
# Colab'da çalıştığını algılar, repo'yu klonlar, eksik IMDb raw dosyasını indirir,
# ve çalışma dizinini notebooks/ klasörüne ayarlar, böylece tüm ../data ve
# ../figures yolları aynen çalışır.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/movie_popularity_analysis"
    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        os.system(f"git clone -q https://github.com/korcanbaykall/movie_popularity_analysis.git {REPO_DIR}")

    # Eksik olabilecek klasörleri garanti et
    os.makedirs(f"{REPO_DIR}/data/raw", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data/processed", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/figures", exist_ok=True)

    # title.basics.tsv.gz repoda yok (~210 MB), IMDb'den indir
    basics_path = f"{REPO_DIR}/data/raw/title.basics.tsv.gz"
    if not os.path.exists(basics_path):
        print("Downloading title.basics.tsv.gz (~210 MB)...")
        os.system(f"wget -q -O {basics_path} https://datasets.imdbws.com/title.basics.tsv.gz")

    # title.ratings.tsv.gz yoksa o da indirilsin
    ratings_path = f"{REPO_DIR}/data/raw/title.ratings.tsv.gz"
    if not os.path.exists(ratings_path):
        print("Downloading title.ratings.tsv.gz...")
        os.system(f"wget -q -O {ratings_path} https://datasets.imdbws.com/title.ratings.tsv.gz")

    # Relative yolların çalışması için notebooks/ dizinine geç
    os.chdir(f"{REPO_DIR}/notebooks")
    print("Working directory:", os.getcwd())

# Create output directories
os.makedirs('../figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

In [ ]:
basics = pd.read_csv(
    "../data/raw/title.basics.tsv.gz",
    sep="\t",
    na_values="\\N",
    low_memory=False
)

ratings = pd.read_csv(
    "../data/raw/title.ratings.tsv.gz",
    sep="\t",
    na_values="\\N",
    low_memory=False
)

print("basics shape:", basics.shape)
print("ratings shape:", ratings.shape)

In [ ]:
basics.head()

In [ ]:
ratings.head()

## 2. Filtering and Merging IMDb Tables

We keep only non-adult movies and join basics with ratings on `tconst` (IMDb's unique title ID).

In [ ]:
movies = basics[basics["titleType"] == "movie"].copy()
movies = movies[movies["isAdult"] == 0]

print("Movies shape:", movies.shape)
movies.head()

In [ ]:
df = movies.merge(ratings, on="tconst", how="left")

df = df[[
    "tconst", "primaryTitle", "startYear",
    "runtimeMinutes", "genres", "averageRating", "numVotes"
]].copy()

print("Merged shape:", df.shape)
df.head()

## 3. Cleaning and Type Conversion

Numeric columns come in as strings. We convert them, then drop rows with missing essential fields and remove duplicates.

In [ ]:
df["startYear"] = pd.to_numeric(df["startYear"], errors="coerce")
df["runtimeMinutes"] = pd.to_numeric(df["runtimeMinutes"], errors="coerce")
df["averageRating"] = pd.to_numeric(df["averageRating"], errors="coerce")
df["numVotes"] = pd.to_numeric(df["numVotes"], errors="coerce")

print("Null counts before cleaning:")
print(df.isnull().sum())

In [ ]:
df = df.dropna(subset=["primaryTitle", "startYear", "genres"])
df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)
df.describe()

## 4. Exploratory Data Analysis — IMDb

We look at the distributions of ratings, vote counts, top genres, and the relationship between rating and number of votes.

In [ ]:
# Distribution of IMDb Average Rating
plt.figure(figsize=(8, 5))
plt.hist(df["averageRating"].dropna(), bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of IMDb Average Rating")
plt.xlabel("Average Rating")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/imdb_rating_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Distribution of Number of Votes
plt.figure(figsize=(8, 5))
plt.hist(df["numVotes"].dropna(), bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of Number of Votes")
plt.xlabel("Number of Votes")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/imdb_votes_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Top 10 Genres
genre_counts = df["genres"].str.split(",").explode().value_counts()
top_genres = genre_counts.head(10)

plt.figure(figsize=(10, 5))
plt.bar(top_genres.index, top_genres.values, edgecolor="black", alpha=0.7)
plt.title("Top 10 Genres")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../figures/top10_genres.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Average Rating vs Number of Votes (scatter)
sample_df = df.dropna(subset=["averageRating", "numVotes"]).sample(
    min(5000, len(df.dropna(subset=["averageRating", "numVotes"]))),
    random_state=42
)

plt.figure(figsize=(8, 5))
plt.scatter(sample_df["averageRating"], sample_df["numVotes"], alpha=0.4, s=10)
plt.title("Average Rating vs Number of Votes")
plt.xlabel("Average Rating")
plt.ylabel("Number of Votes")
plt.tight_layout()
plt.savefig("../figures/rating_vs_votes.png", dpi=150, bbox_inches="tight")
plt.show()

### Save cleaned IMDb dataset

In [ ]:
df.to_csv("../data/processed/movies_imdb_cleaned.csv", index=False)
print("Saved movies_imdb_cleaned.csv — shape:", df.shape)

## 5. Data Enrichment with TMDb API

Per the project requirements, we enrich the IMDb data with a second source. We sample 2 000 movies, look each one up on TMDb by its IMDb ID, and fetch popularity, original language, release date, TMDb vote average, and TMDb genres.

> **Note:** The API fetching code below takes ~30 minutes. The pre-fetched result is saved at `../data/processed/movies_imdb_tmdb_2000.csv`. If that file already exists, the cell below loads it directly instead of re-fetching.

In [ ]:
TMDB_CSV = "../data/processed/movies_imdb_tmdb_2000.csv"

if os.path.exists(TMDB_CSV):
    merged_tmdb = pd.read_csv(TMDB_CSV)
    print("Loaded cached TMDb data from CSV — shape:", merged_tmdb.shape)
else:
    # ── If you need to re-fetch, uncomment the block below and supply your token ──
    # import requests
    # from getpass import getpass
    #
    # TMDB_BEARER_TOKEN = getpass("TMDb API Read Access Token: ")
    # headers = {
    #     "Authorization": f"Bearer {TMDB_BEARER_TOKEN}",
    #     "accept": "application/json"
    # }
    #
    # tmdb_sample = df.dropna(subset=["tconst"]).sample(
    #     min(2000, len(df.dropna(subset=["tconst"]))), random_state=42
    # ).copy()
    #
    # def find_tmdb_movie_by_imdb(imdb_id):
    #     url = f"https://api.themoviedb.org/3/find/{imdb_id}"
    #     params = {"external_source": "imdb_id"}
    #     response = requests.get(url, headers=headers, params=params, timeout=30)
    #     if response.status_code != 200:
    #         return None
    #     data = response.json()
    #     results = data.get("movie_results", [])
    #     return results[0]["id"] if results else None
    #
    # def get_tmdb_movie_details(tmdb_id, imdb_id):
    #     url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    #     response = requests.get(url, headers=headers, timeout=30)
    #     if response.status_code != 200:
    #         return {"tconst": imdb_id, "tmdb_id": tmdb_id, "popularity": None,
    #                 "original_language": None, "release_date": None,
    #                 "tmdb_vote_average": None, "tmdb_vote_count": None, "tmdb_genres": None}
    #     d = response.json()
    #     return {
    #         "tconst": imdb_id, "tmdb_id": tmdb_id,
    #         "popularity": d.get("popularity"),
    #         "original_language": d.get("original_language"),
    #         "release_date": d.get("release_date"),
    #         "tmdb_vote_average": d.get("vote_average"),
    #         "tmdb_vote_count": d.get("vote_count"),
    #         "tmdb_genres": ",".join([g["name"] for g in d.get("genres", [])])
    #     }
    #
    # import time
    # tmdb_rows = []
    # for i, imdb_id in enumerate(tmdb_sample["tconst"], start=1):
    #     tid = find_tmdb_movie_by_imdb(imdb_id)
    #     if tid is None:
    #         tmdb_rows.append({"tconst": imdb_id, "tmdb_id": None, "popularity": None,
    #                           "original_language": None, "release_date": None,
    #                           "tmdb_vote_average": None, "tmdb_vote_count": None, "tmdb_genres": None})
    #     else:
    #         tmdb_rows.append(get_tmdb_movie_details(tid, imdb_id))
    #     if i % 100 == 0:
    #         print(f"Fetched {i} / {len(tmdb_sample)}")
    #     time.sleep(0.25)
    #
    # tmdb_df = pd.DataFrame(tmdb_rows)
    # merged_tmdb = tmdb_sample.merge(tmdb_df, on="tconst", how="left")
    # merged_tmdb.to_csv(TMDB_CSV, index=False)
    print("ERROR: TMDb CSV not found. Please run the API fetch code above.")

merged_tmdb.head()

## 6. Exploratory Data Analysis — Merged IMDb + TMDb

Now we explore TMDb popularity on its own and in relation to IMDb features.

In [ ]:
# Distribution of TMDb Popularity
plt.figure(figsize=(8, 5))
plt.hist(merged_tmdb["popularity"].dropna(), bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of TMDb Popularity")
plt.xlabel("Popularity")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/tmdb_popularity_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# IMDb Average Rating vs TMDb Popularity
plot_df = merged_tmdb.dropna(subset=["averageRating", "popularity"]).copy()

plt.figure(figsize=(8, 5))
plt.scatter(plot_df["averageRating"], plot_df["popularity"], alpha=0.4, s=10)
plt.title("IMDb Average Rating vs TMDb Popularity")
plt.xlabel("IMDb Average Rating")
plt.ylabel("TMDb Popularity")
plt.tight_layout()
plt.savefig("../figures/rating_vs_popularity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# IMDb Number of Votes vs TMDb Popularity
plot_df = merged_tmdb.dropna(subset=["numVotes", "popularity"]).copy()

plt.figure(figsize=(8, 5))
plt.scatter(plot_df["numVotes"], plot_df["popularity"], alpha=0.4, s=10)
plt.title("IMDb Number of Votes vs TMDb Popularity")
plt.xlabel("IMDb Number of Votes")
plt.ylabel("TMDb Popularity")
plt.tight_layout()
plt.savefig("../figures/votes_vs_popularity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Top 10 Original Languages
top_lang = merged_tmdb["original_language"].value_counts().head(10)

plt.figure(figsize=(10, 5))
plt.bar(top_lang.index, top_lang.values, edgecolor="black", alpha=0.7)
plt.title("Top 10 Original Languages")
plt.xlabel("Language")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../figures/top10_languages.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Additional EDA — Log Scale and Genre Comparison

The `popularity` and `numVotes` distributions are heavily right-skewed, so linear-scale histograms are hard to read. We add log-scale versions and compare popularity across the top genres.

In [ ]:
# Log-scale popularity histogram
pop = merged_tmdb["popularity"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(np.log1p(pop), bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of TMDb Popularity (log scale)")
plt.xlabel("log(1 + popularity)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/popularity_log_hist.png", dpi=150, bbox_inches="tight")
plt.show()

The log-scale plot makes the distribution much easier to read — most movies have low popularity, with a long right tail of very popular ones.

In [ ]:
# Genre vs popularity boxplot (use first listed genre for each movie)
df_box = merged_tmdb.dropna(subset=["popularity", "genres"]).copy()
df_box["first_genre"] = df_box["genres"].str.split(",").str[0]
top_genres_list = df_box["first_genre"].value_counts().head(8).index.tolist()
data_by_genre = [df_box.loc[df_box["first_genre"] == g, "popularity"].values for g in top_genres_list]

plt.figure(figsize=(10, 5))
bp = plt.boxplot(data_by_genre, showfliers=False)
plt.xticks(range(1, len(top_genres_list) + 1), top_genres_list, rotation=45)
plt.title("TMDb Popularity by Primary Genre")
plt.xlabel("Primary Genre")
plt.ylabel("Popularity")
plt.tight_layout()
plt.savefig("../figures/genre_popularity_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

The boxplot suggests that popularity varies across genres — Action and Adventure tend to have higher medians than Documentary or Drama. We test this formally below.

## 8. Hypothesis Tests

We run three hypothesis tests. Because `popularity` is strongly right-skewed, we use **non-parametric tests** (Mann-Whitney U, Kruskal-Wallis, Spearman correlation) rather than t-tests and Pearson correlation, which assume approximate normality.

Significance level: **α = 0.05** for all tests.

In [ ]:
# Prepare a clean dataframe for the tests
test_df = merged_tmdb.dropna(subset=["popularity", "genres", "averageRating"]).copy()
print("Rows used in tests:", len(test_df))

### Test A — Are Action movies more popular than non-Action movies?

- **H₀:** The popularity distributions of Action and non-Action movies are the same.
- **H₁:** They differ.
- **Test:** Mann-Whitney U (two-sided).

In [ ]:
test_df["is_action"] = test_df["genres"].str.contains("Action", na=False)
action = test_df.loc[test_df["is_action"], "popularity"]
non_action = test_df.loc[~test_df["is_action"], "popularity"]

u_stat, p_value = stats.mannwhitneyu(action, non_action, alternative="two-sided")

print(f"Action movies:     n={len(action):4d}  median popularity={action.median():.3f}")
print(f"Non-Action movies: n={len(non_action):4d}  median popularity={non_action.median():.3f}")
print(f"U statistic = {u_stat:.0f}")
print(f"p-value     = {p_value:.4g}")
print()
if p_value < 0.05:
    print("Result: Reject H0 -> Action movies have significantly different popularity.")
else:
    print("Result: Fail to reject H0 -> No significant difference.")

### Test B — Does popularity differ across the top genres?

- **H₀:** All top-5 genres have the same popularity distribution.
- **H₁:** At least one genre has a different popularity distribution.
- **Test:** Kruskal-Wallis (non-parametric one-way ANOVA).

In [ ]:
test_df["first_genre"] = test_df["genres"].str.split(",").str[0]
top5 = test_df["first_genre"].value_counts().head(5).index.tolist()
groups = [test_df.loc[test_df["first_genre"] == g, "popularity"].values for g in top5]

h_stat, p_value = stats.kruskal(*groups)

print("Top 5 primary genres tested:", top5)
for g, vals in zip(top5, groups):
    print(f"  {g:12s} n={len(vals):4d}  median={pd.Series(vals).median():.3f}")
print()
print(f"H statistic = {h_stat:.3f}")
print(f"p-value     = {p_value:.4g}")
print()
if p_value < 0.05:
    print("Result: Reject H0 -> Popularity differs significantly across genres.")
else:
    print("Result: Fail to reject H0 -> No significant difference.")

### Test C — Is IMDb rating correlated with TMDb popularity?

- **H₀:** No monotonic relationship between IMDb average rating and TMDb popularity (ρ = 0).
- **H₁:** There is a monotonic relationship (ρ ≠ 0).
- **Test:** Spearman rank correlation (robust to skew and non-linearity).

In [ ]:
rho, p_value = stats.spearmanr(test_df["averageRating"], test_df["popularity"])

print(f"Spearman rho = {rho:.4f}")
print(f"p-value      = {p_value:.4g}")
print()
if p_value < 0.05:
    direction = "positive" if rho > 0 else "negative"
    strength = "strong" if abs(rho) > 0.5 else ("moderate" if abs(rho) > 0.3 else "weak")
    print(f"Result: Reject H0 -> {strength} {direction} correlation between rating and popularity.")
else:
    print("Result: Fail to reject H0 -> No significant correlation.")

## 9. Summary of Findings

- The merged IMDb + TMDb dataset contains about 1 250 movies with non-null popularity after enrichment.
- TMDb popularity is heavily right-skewed; a log transform is needed for visualisation.
- **Action movies are significantly more popular** than non-Action movies (Mann-Whitney, p < 0.001).
- **Popularity differs significantly across top genres** (Kruskal-Wallis, p < 0.001).
- **IMDb rating and TMDb popularity are weakly negatively correlated** (Spearman ρ ≈ −0.14, p < 0.001), meaning higher-rated movies tend to be *slightly* less popular on TMDb — possibly because critically-acclaimed niche films receive high ratings but lower mainstream popularity.

### Next Steps (for upcoming milestones)
- Apply ML models (e.g. Random Forest, Gradient Boosting) to predict popularity from genre, language, rating, and year.
- Perform feature engineering (one-hot encode genres, decade bins, etc.).
- Evaluate model performance and report final findings.